# COME 448 - Duygu Analizi (Sentiment Analysis)
## 3 Farklı Sınıflandırıcı ile Karşılaştırma

**Veri Seti:** Turkish Product Reviews

**Modeller:**
1. BERTurk (`dbmdz/bert-base-turkish-cased`)
2. DistilBERTurk (`dbmdz/distilbert-base-turkish-cased`)
3. XLM-RoBERTa (`xlm-roberta-base`)

## 1. Kütüphanelerin Yüklenmesi

In [ ]:
!pip install transformers datasets accelerate -q

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report, roc_curve, auc
)
warnings.filterwarnings('ignore')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Yapılandırma

In [ ]:
MODELS = {
    'BERTurk': 'dbmdz/bert-base-turkish-cased',
    'DistilBERTurk': 'dbmdz/distilbert-base-turkish-cased',
    'XLM-RoBERTa': 'xlm-roberta-base'
}
MAX_LENGTH = 128
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
WARMUP_STEPS = 500
WEIGHT_DECAY = 0.01
TRAIN_SIZE = 10000
TEST_SIZE = 2000
RESULTS_DIR = 'results'
os.makedirs(RESULTS_DIR, exist_ok=True)

## 3. Veri Seti Yükleme ve Ön İşleme

In [ ]:
dataset = load_dataset('fthbrmnby/turkish_product_reviews')
df = pd.DataFrame(dataset['train'])
print(f'Toplam veri: {len(df)}')
print(f'Sütunlar: {df.columns.tolist()}')
df.head()

In [ ]:
print('Etiket dagilimi:')
print(df['sentiment'].value_counts())

label_map = {'positive': 1, 'negative': 0}
if df['sentiment'].dtype == object:
    df['label'] = df['sentiment'].map(label_map)
else:
    df['label'] = df['sentiment']

text_col = 'sentence' if 'sentence' in df.columns else df.columns[0]
df['text'] = df[text_col].astype(str)
df = df[df['text'].str.strip().str.len() > 0].reset_index(drop=True)

# Sinif dengeleme (undersampling)
df_pos = df[df['label'] == 1]
df_neg = df[df['label'] == 0]
print(f'\nDengeleme oncesi - Pozitif: {len(df_pos)}, Negatif: {len(df_neg)}')

train_per_class = TRAIN_SIZE // 2  # Her siniftan 5000
test_per_class = TEST_SIZE // 2    # Her siniftan 1000

df_pos = df_pos.sample(frac=1, random_state=42).reset_index(drop=True)
df_neg = df_neg.sample(frac=1, random_state=42).reset_index(drop=True)

train_df = pd.concat([df_pos.head(train_per_class), df_neg.head(train_per_class)]).sample(frac=1, random_state=42).reset_index(drop=True)
test_df = pd.concat([df_pos.iloc[train_per_class:train_per_class+test_per_class], df_neg.iloc[train_per_class:train_per_class+test_per_class]]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f'\nEgitim: {len(train_df)}, Test: {len(test_df)}')
print(f'Egitim etiket dagilimi:\n{train_df["label"].value_counts()}')
print(f'Test etiket dagilimi:\n{test_df["label"].value_counts()}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
labels_str = ['Negatif', 'Pozitif']
tr_c = [len(train_df[train_df['label']==0]), len(train_df[train_df['label']==1])]
te_c = [len(test_df[test_df['label']==0]), len(test_df[test_df['label']==1])]
x = np.arange(2); w = 0.35
axes[0].bar(x-w/2, tr_c, w, label='Egitim', color='#4C72B0')
axes[0].bar(x+w/2, te_c, w, label='Test', color='#DD8452')
axes[0].set_xticks(x); axes[0].set_xticklabels(labels_str)
axes[0].set_title('Etiket Dagilimi'); axes[0].legend()

tr_len = train_df['text'].apply(lambda x: len(x.split()))
axes[1].hist(tr_len, bins=50, alpha=0.7, color='#4C72B0')
axes[1].set_title('Metin Uzunlugu Dagilimi'); axes[1].set_xlabel('Kelime Sayisi')

tmp = train_df.copy()
tmp['kelime'] = tr_len
tmp['duygu'] = tmp['label'].map({0:'Negatif',1:'Pozitif'})
sns.boxplot(data=tmp, x='duygu', y='kelime', ax=axes[2], palette=['#DD8452','#4C72B0'])
axes[2].set_title('Duyguya Gore Metin Uzunlugu')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/veri_seti_istatistikleri.png', dpi=150)
plt.show()

## 4. Dataset Sinifi ve Yardimci Fonksiyonlar

In [ ]:
class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=MAX_LENGTH):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tokenizer(str(self.texts[idx]), max_length=self.max_length,
                             padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].flatten(),
                'attention_mask': enc['attention_mask'].flatten(),
                'labels': torch.tensor(int(self.labels[idx]), dtype=torch.long)}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    return {'accuracy': accuracy_score(labels, preds), 'f1': f1, 'precision': prec, 'recall': rec}

## 5. Model Egitim Fonksiyonu

In [ ]:
def train_model(model_name, model_path, train_df, test_df):
    print(f'\n{"="*50}\nMODEL: {model_name} ({model_path})\n{"="*50}')
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    train_ds = SentimentDataset(train_df['text'].tolist(), train_df['label'].tolist(), tokenizer)
    test_ds = SentimentDataset(test_df['text'].tolist(), test_df['label'].tolist(), tokenizer)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_path, num_labels=2,
        id2label={0:'negative',1:'positive'}, label2id={'negative':0,'positive':1})
    out_dir = f'{RESULTS_DIR}/ckpt_{model_name.lower().replace("-","_")}'
    args = TrainingArguments(
        output_dir=out_dir, num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE*2,
        warmup_steps=WARMUP_STEPS, weight_decay=WEIGHT_DECAY, learning_rate=LEARNING_RATE,
        logging_steps=100, eval_strategy='epoch', save_strategy='epoch',
        load_best_model_at_end=True, metric_for_best_model='f1',
        fp16=torch.cuda.is_available(), report_to='none', seed=42)
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=test_ds,
                      compute_metrics=compute_metrics,
                      callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    start = time.time()
    trainer.train()
    t_time = time.time() - start
    print(f'Egitim suresi: {t_time:.1f}s')
    ev = trainer.evaluate()
    pred_out = trainer.predict(test_ds)
    preds = np.argmax(pred_out.predictions, axis=-1)
    probs = torch.softmax(torch.tensor(pred_out.predictions), dim=-1).numpy()
    print(classification_report(test_df['label'].tolist(), preds, target_names=['Negatif','Pozitif']))
    import shutil
    if os.path.exists(out_dir): shutil.rmtree(out_dir)
    return {'model_name': model_name, 'accuracy': ev['eval_accuracy'],
            'precision': ev['eval_precision'], 'recall': ev['eval_recall'],
            'f1': ev['eval_f1'], 'training_time': t_time,
            'predictions': preds, 'probabilities': probs,
            'true_labels': test_df['label'].tolist()}

## 6. Modellerin Egitilmesi

In [ ]:
all_results = []
for name, path in MODELS.items():
    try:
        result = train_model(name, path, train_df, test_df)
        all_results.append(result)
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    except Exception as e:
        print(f'HATA {name}: {e}')
print(f'\nBasarili model sayisi: {len(all_results)}')

## 7. Sonuclar ve Gorsellestirme
### 7.1 Ozet Tablo

In [ ]:
summary = pd.DataFrame([{
    'Model': r['model_name'], 'Accuracy': f"{r['accuracy']:.4f}",
    'Precision': f"{r['precision']:.4f}", 'Recall': f"{r['recall']:.4f}",
    'F1-Score': f"{r['f1']:.4f}", 'Sure (s)': f"{r['training_time']:.1f}"
} for r in all_results])
summary.to_csv(f'{RESULTS_DIR}/sonuc_tablosu.csv', index=False, encoding='utf-8-sig')
summary

### 7.2 Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1, len(all_results), figsize=(6*len(all_results), 5))
if len(all_results) == 1: axes = [axes]
for i, r in enumerate(all_results):
    cm = confusion_matrix(r['true_labels'], r['predictions'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Negatif','Pozitif'], yticklabels=['Negatif','Pozitif'], ax=axes[i])
    axes[i].set_title(f"{r['model_name']}\n(Acc: {r['accuracy']:.4f})")
    axes[i].set_ylabel('Gercek'); axes[i].set_xlabel('Tahmin')
plt.suptitle('Confusion Matrix Karsilastirmasi', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/confusion_matrices.png', dpi=150)
plt.show()

### 7.3 Metrik Karsilastirma

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
names = [r['model_name'] for r in all_results]
metrics = ['accuracy','precision','recall','f1']
mlabels = ['Accuracy','Precision','Recall','F1-Score']
colors = ['#4C72B0','#DD8452','#55A868','#C44E52']
x = np.arange(len(names)); w = 0.2
for i,(m,ml,c) in enumerate(zip(metrics,mlabels,colors)):
    vals = [r[m] for r in all_results]
    bars = axes[0].bar(x+i*w, vals, w, label=ml, color=c)
    for b,v in zip(bars,vals):
        axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+0.005, f'{v:.3f}', ha='center', fontsize=8)
axes[0].set_xticks(x+w*1.5); axes[0].set_xticklabels(names)
axes[0].set_title('Model Performans Karsilastirmasi'); axes[0].legend(); axes[0].set_ylim(0,1.1)
times = [r['training_time'] for r in all_results]
bars = axes[1].bar(names, times, color=colors[:len(names)])
for b,v in zip(bars,times):
    axes[1].text(b.get_x()+b.get_width()/2, b.get_height()+1, f'{v:.0f}s', ha='center')
axes[1].set_title('Egitim Suresi Karsilastirmasi'); axes[1].set_ylabel('Sure (s)')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/metrik_karsilastirma.png', dpi=150)
plt.show()

### 7.4 ROC Egrisi

In [ ]:
plt.figure(figsize=(10, 8))
colors = ['#4C72B0','#DD8452','#55A868']
for i, r in enumerate(all_results):
    fpr, tpr, _ = roc_curve(r['true_labels'], r['probabilities'][:,1])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=colors[i], lw=2, label=f"{r['model_name']} (AUC={roc_auc:.4f})")
plt.plot([0,1],[0,1],'gray',ls='--',label='Rastgele (AUC=0.5)')
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Egrisi Karsilastirmasi', fontweight='bold')
plt.legend(loc='lower right'); plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/roc_curves.png', dpi=150)
plt.show()

### 7.5 Son Analiz

In [ ]:
best_f1 = max(all_results, key=lambda x: x['f1'])
best_acc = max(all_results, key=lambda x: x['accuracy'])
fastest = min(all_results, key=lambda x: x['training_time'])
print(f"En yuksek F1-Score: {best_f1['model_name']} ({best_f1['f1']:.4f})")
print(f"En yuksek Accuracy: {best_acc['model_name']} ({best_acc['accuracy']:.4f})")
print(f"En hizli egitim: {fastest['model_name']} ({fastest['training_time']:.1f}s)")
print('\n--- Model Bazinda Sonuclar ---')
for r in all_results:
    print(f"\n{r['model_name']}: Acc={r['accuracy']:.4f} F1={r['f1']:.4f} "
          f"Prec={r['precision']:.4f} Rec={r['recall']:.4f} Sure={r['training_time']:.1f}s")